In [7]:

!pip -q install keras-tuner

import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

(x, y), (xt, yt) = imdb.load_data(num_words=8000)

x = pad_sequences(x, maxlen=100)
xt = pad_sequences(xt, maxlen=100)

def build_model(hp):

    cell = hp.Choice(
        "cell", ["SimpleRNN", "LSTM", "GRU"])

    units = hp.Choice("units", [32, 64])
    dropout = hp.Choice("dropout", [0.2, 0.4])

    inp = tf.keras.Input((100,))
    z = tf.keras.layers.Embedding(8000, 32)(inp)

    layer = {
        "SimpleRNN": tf.keras.layers.SimpleRNN,
        "LSTM": tf.keras.layers.LSTM,
        "GRU": tf.keras.layers.GRU
    }[cell]

    z = layer(units)(z)
    z = tf.keras.layers.Dropout(dropout)(z)

    out = tf.keras.layers.Dense(
        1, activation="sigmoid")(z)

    model = tf.keras.Model(inp, out)

    model.compile(
        "adam",
        "binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=6,
    directory="tuner",
    project_name="rnn_lstm_gru",
    overwrite=True
)

tuner.search(
    x[:10000], y[:10000],
    epochs=3,
    validation_split=0.2,
    batch_size=128
)

print("Best Parameters:")
print(tuner.get_best_hyperparameters(1)[0].values)

best_model = tuner.get_best_models(1)[0]

print("Test Results:")
print(best_model.evaluate(xt, yt))


Trial 6 Complete [00h 00m 15s]
val_accuracy: 0.8324999809265137

Best val_accuracy So Far: 0.8399999737739563
Total elapsed time: 00h 02m 26s
Best Parameters:
{'cell': 'LSTM', 'units': 32, 'dropout': 0.4}


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Test Results:
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.8329 - loss: 0.3798
[0.3798353970050812, 0.8329200148582458]
